# Google Cloud Vertex AI

Vertex AI는 Google의 통합 ML 플랫폼입니다.

## 학습 목표
- Vertex AI 프로젝트 설정
- 모델 학습 및 배포
- 엔드포인트 관리

## 1. 환경 설정

In [ ]:
# Google Cloud SDK 설치 필요
# pip install google-cloud-aiplatform

# from google.cloud import aiplatform

# aiplatform.init(
#     project='your-project-id',
#     location='us-central1'
# )

print("Vertex AI 초기화 준비!")
print("\n사전 준비:")
print("1. Google Cloud 프로젝트 생성")
print("2. Vertex AI API 활성화")
print("3. 서비스 계정 키 생성")
print("4. gcloud auth login")

## 2. 데이터 준비

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 샘플 데이터 생성
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(10)])
df['target'] = y

# CSV로 저장
df.to_csv('training_data.csv', index=False)
print("데이터 준비 완료!")
print(df.head())

## 3. Custom Training Job

In [ ]:
# 학습 스크립트 생성
training_script = """
import argparse
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators', type=int, default=100)
    parser.add_argument('--max-depth', type=int, default=10)
    args = parser.parse_args()
    
    # 데이터 로딩
    df = pd.read_csv('/gcs/your-bucket/training_data.csv')
    X = df.drop('target', axis=1)
    y = df['target']
    
    # 학습/테스트 분할
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    
    # 모델 학습
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth
    )
    model.fit(X_train, y_train)
    
    # 평가
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f'정확도: {accuracy}')
    
    # 모델 저장
    joblib.dump(model, '/gcs/your-bucket/model.pkl')

if __name__ == '__main__':
    main()
"""

with open('train.py', 'w') as f:
    f.write(training_script)

print("학습 스크립트 생성 완료!")

## 4. Vertex AI SDK 실행

In [ ]:
# Vertex AI Custom Training Job 실행 예시
# 실제 실행시 아래 코드의 주석을 해제하세요

vertex_ai_code = """
from google.cloud import aiplatform

# 초기화
aiplatform.init(project='your-project-id', location='us-central1')

# Custom Training Job 생성
job = aiplatform.CustomTrainingJob(
    display_name='random-forest-training',
    script_path='train.py',
    container_uri='us-docker.pkg.dev/vertex-ai/training/sklearn-gpu.1-0:latest',
    requirements=['pandas', 'scikit-learn']
)

# 모델 학습 실행
model = job.run(
    replica_count=1,
    machine_type='n1-standard-4',
    args=['--n-estimators', '100', '--max-depth', '10']
)

# 엔드포인트에 배포
endpoint = model.deploy(
    deployed_model_display_name='random-forest-endpoint',
    machine_type='n1-standard-4'
)

print(f'엔드포인트: {endpoint.resource_name}')
"""

print("Vertex AI 실행 코드:")
print(vertex_ai_code)

## 5. 예측 요청

In [ ]:
# 예측 요청 예시
prediction_code = """
from google.cloud import aiplatform

# 엔드포인트 로딩
endpoint = aiplatform.Endpoint('projects/your-project/locations/us-central1/endpoints/your-endpoint-id')

# 예측 요청
instances = [[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]]
prediction = endpoint.predict(instances=instances)

print(f'예측 결과: {prediction.predictions}')
"""

print("예측 요청 코드:")
print(prediction_code)

## Vertex AI 주요 기능

| 기능 | 설명 |
|------|------|
| Custom Training | 컨테이너 기반 모델 학습 |
| AutoML | 자동 모델 학습 |
| Model Garden | 사전 학습된 모델 허브 |
| Feature Store | 특성 관리 |
| Pipelines | ML 파이프라인 자동화 |
| Experiments | 실험 추적 |